# Temperature Forecasting

## Project Overview

Forecasts **daily average temperature** (°C) over a 14-day horizon. Synthetic data spans 2 years with strong annual seasonality, day-to-day weather variability, and a slight warming trend.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily temperatures, predict the next 14 days. Temperature forecasts drive energy demand planning, agriculture scheduling, and public health advisories.

## Why This Project Matters

Temperature is the single most impactful weather variable. It drives heating/cooling demand, crop growth, disease vectors, and infrastructure stress. Accurate medium-range temperature forecasts enable proactive planning across energy, agriculture, and public health sectors.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily average temperature
- Strong annual cycle (summer ~30°C, winter ~5°C)
- Day-to-day weather variability
- Slight warming trend
- No weekly pattern (weather doesn't know weekdays)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `temp_c` | Daily average temperature (°C) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'temp_c'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

# Strong annual cycle
seasonal = 17.5 + 12.5 * np.sin(2 * np.pi * (t - 80) / 365)
trend = 0.002 * t  # slight warming

# Day-to-day weather variability (autocorrelated noise)
weather = np.zeros(N_POINTS)
weather[0] = np.random.normal(0, 2)
for i in range(1, N_POINTS):
    weather[i] = 0.6 * weather[i-1] + np.random.normal(0, 2)

temp_c = seasonal + trend + weather
temp_c = np.round(temp_c, 1)
# Ensure positive for validation (shift if needed)
temp_c = temp_c - temp_c.min() + 1.0

df = pd.DataFrame({'date': dates, 'temp_c': temp_c})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,temp_c
0,2022-01-01,6.8
1,2022-01-02,6.2
2,2022-01-03,7.4
3,2022-01-04,9.9
4,2022-01-05,7.9


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('temp_c Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("temp_c Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

temp_c Statistics:
count    730.000000
mean      18.762466
std        9.144508
min        1.000000
25%       10.200000
50%       18.600000
75%       27.375000
max       37.300000
Name: temp_c, dtype: float64

CV: 0.487
Skewness: 0.007


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:        2.7 | RMSE:        3.1 | MAPE: 56.33%
Seasonal Naive (7)             | MAE:        1.5 | RMSE:        2.4 | MAPE: 38.94%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                            Adjusted R-Squared  R-Squared       RMSE  Time Taken
Model                                                                           
KernelRidge                        1254.937440 -95.456726  20.384714    0.057094
QuantileRegressor                   575.076933 -43.159764  13.792778    0.066219
DummyRegressor                      550.815208 -41.293478  13.498175    0.007026
PassiveAggressiveRegressor           84.767299  -5.443638   5.268706    0.009942
Lars                                 38.231911  -1.863993   3.512565    0.012716
DecisionTreeRegressor                34.198290  -1.553715   3.316840    0.010864
GammaRegressor                       29.928769  -1.225290   3.096219    4.706279
LassoLars                            28.385269  -1.106559   3.012487    0.009704
Lasso                                28.358928  -1.104533   3.011038    0.011442
ElasticNet                           27.385180  -1.029629   2.956969    0.007887


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: lgbm
FLAML (lgbm)                   | MAE:        1.8 | RMSE:        2.2 | MAPE: 36.84%
FLAML Test (lgbm)              | MAE:        1.1 | RMSE:        1.5 | MAPE: 23.75%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:        1.5 | RMSE:        2.1 | MAPE: 36.24%
SF AutoETS                     | MAE:        2.2 | RMSE:        2.7 | MAPE: 48.16%
SF SeasonalNaive               | MAE:        1.6 | RMSE:        2.5 | MAPE: 40.34%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
             model      MAE     RMSE      MAPE
 FLAML Test (lgbm) 1.073338 1.486421 23.752924
      SF AutoARIMA 1.527507 2.121654 36.242442
Seasonal Naive (7) 1.550000 2.404906 38.936648
  SF SeasonalNaive 1.621429 2.492990 40.340616
      FLAML (lgbm) 1.825101 2.168232 36.842334
        SF AutoETS 2.188574 2.704363 48.159907
Naive (Last Value) 2.664286 3.101958 56.332337

Top 3:
             model      MAE     RMSE      MAPE
 FLAML Test (lgbm) 1.073338 1.486421 23.752924
      SF AutoARIMA 1.527507 2.121654 36.242442
Seasonal Naive (7) 1.550000 2.404906 38.936648


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -0.25, Std: 1.46


## Interpretation and Business Insight

- Temperature has the strongest annual seasonality of any weather variable
- Day-to-day variability is autocorrelated (warm days tend to follow warm days)
- No weekly pattern exists (purely driven by solar angle and weather systems)
- Slight warming trend is consistent with climate change
- Seasonal naive baseline is surprisingly effective

## Limitations

1. Synthetic — real temperature depends on geography, elevation, ocean currents
2. No atmospheric pressure, humidity, or wind data
3. Single location — real forecasting needs spatial models
4. Daily average misses diurnal range (min/max)
5. No extreme event modeling (heat waves, cold snaps)

## How to Improve This Project

1. Add atmospheric variables (pressure, humidity, wind)
2. Use NWP (numerical weather prediction) model output
3. Model min/max temperature separately
4. Include spatial data from nearby stations
5. Add climate indices (ENSO, NAO) for seasonal outlooks

## Production Considerations

- Daily temperature forecasts for energy demand planning
- Integration with NWP ensemble models
- Heat wave / cold snap early warning systems
- Agricultural frost risk alerting

## Common Mistakes

1. Expecting weekly patterns in temperature (there are none)
2. Not using autocorrelated noise models (AR processes)
3. Ignoring the diurnal range (daily avg hides important info)
4. Using statistical models beyond 10-14 days (NWP needed)

## Mini Challenge / Exercises

1. Decompose into trend + seasonal + residual — analyze each
2. Fit an AR(1) model to the residuals and compare
3. Build a heat wave detector (consecutive days above threshold)
4. Compare MAPE at 1-day vs 7-day vs 14-day horizons

## Final Summary / Key Takeaways

1. Temperature forecasting is fundamental to weather-dependent industries
2. Annual seasonality is dominant and highly predictable
3. Day-to-day variability is autocorrelated (persistence matters)
4. Statistical models work well up to ~10 days; beyond that NWP is needed
5. Extreme event detection is the highest-value application

In [18]:
import json
metrics = {
    'project': 'Temperature Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Temperature Forecasting — Complete ===")

Metrics saved.

=== Temperature Forecasting — Complete ===
